In [1]:
import source.data
import source.predict
import source.analysis
import os
import subprocess
import biolib
from pathlib import Path
from Bio import SeqIO
import pandas as pd

C:\Users\domin\anaconda3\envs\GLUT\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
source.data.import_alphamissense()

AlphaMissense data already exists


In [3]:
#pip install pymissense
#pip3 install --upgrade pybiolib 
#pip install gunicorn

In [4]:
RES_DIR = Path.cwd() / 'results'

In [5]:
proteins = [
'P11166',
'P11168',
'P11169',
#'P14672',
#'P22732',
#'Q9UGQ3',
#'Q6PXP3',
#'Q9NY64',
#'Q9NRM0',
#'O95528',
#'Q9BYW1',
#'Q8TD20',
#'Q96QE2',
#'Q8TDB8' 
]

In [6]:
#import pdb and fasta files
for up_id in proteins:
    source.data.import_pdb(up_id)
    source.data.import_fasta(up_id)

In [7]:
# run pymissense for all proteins
for up_id in proteins:
    print(f'running PyMissense for {up_id}')
    source.predict.pymissense(up_id)

running PyMissense for P11166
AlphaMissense prediction for P11166 already exists
running PyMissense for P11168
AlphaMissense prediction for P11168 already exists
running PyMissense for P11169
AlphaMissense prediction for P11169 already exists


In [8]:
#get average pathogenicity per protein

with open(RES_DIR / 'analysis/avg_patho_proteins_alphamissense.csv', 'w') as avg_f:
    for up_id in proteins:
        avg = str(source.analysis.alphamissense_avg_pat(up_id,verbose=True))
        avg_f.write(f'{up_id},{avg}\n')
    

# Result will be writen down to the single AlpaMissense patogenicity.csv file. For each protein will be a number of average pathogenicity.
# Results can be find here /GLUT project documentation/Results from AlphaMissense/AlpaMissense patogenicity.txt/

Average aa pathogenicity for P11166: 0.64
Average aa pathogenicity for P11168: 0.48
Average aa pathogenicity for P11169: 0.62


In [9]:
#get annotations of AA positions relative to the membrane using 
deeptmhmm = biolib.load('DTU/DeepTMHMM')
for up_id in proteins:
    source.predict.deepTMHMM(up_id,deeptmhmm)

2025-08-26 12:20:03,963 | INFO : Loaded project DTU/DeepTMHMM:1.0.44
DeepTMHMM prediction for P11166 already exists
DeepTMHMM prediction for P11168 already exists
DeepTMHMM prediction for P11169 already exists


In [10]:
# Protein regions - processing of results
deepTMHMM3_res = Path.cwd() / 'results/deeptmhmm'

for up_id in proteins:
    source.analysis.deepTMHMM3line_to_csv(input_file=deepTMHMM3_res / f'{up_id}.3line',output_file= deepTMHMM3_res / f'{up_id}.csv')

# Saved in /GLUT project documentation/Results from DeepTMHMM/Excel output from sequences
# After processing Excel files, prepare in files column for Pathogenicity

The resulting file was saved as: C:\Users\domin\Desktop\GLUT\results\deeptmhmm\P11166.csv
The resulting file was saved as: C:\Users\domin\Desktop\GLUT\results\deeptmhmm\P11168.csv
The resulting file was saved as: C:\Users\domin\Desktop\GLUT\results\deeptmhmm\P11169.csv


In [11]:
#Protein regions - pathogenicity assignment. The cell have to be run for each protein separately. 


res = Path.cwd() / 'results'

for up_id in proteins:
    df = source.analysis.region_pathogenicity(region_file=res / f'deeptmhmm/{up_id}.csv', pdb_file= res / f'pymissense/{up_id}-edit.pdb' )

    # Output
    df.to_csv(f"results/analysis/region_pathogenicity_PyMissenseMIO_{up_id}.csv", index=False)
    #df.head()

# Outputs saved in /GLUT project documentation/Results from DeepTMHMM/PyMissense/

In [12]:
#average M(embrane), I(ntracelular), O(extracelular) aa pathogenicity values
uniprot_ids = []
O_pathogenicities = []
M_pathogenicities = []
I_pathogenicities = []

for up_id in proteins:
    file_path = Path.cwd() / f'results/analysis/region_pathogenicity_PyMissenseMIO_{up_id}.csv'
    prot_df = pd.read_csv(file_path)
    uniprot_ids.append(up_id)
    O_pathogenicities.append(prot_df['O_pathogenicity'].mean())
    M_pathogenicities.append(prot_df['M_pathogenicity'].mean())
    I_pathogenicities.append(prot_df['I_pathogenicity'].mean())

avgs = {'uniprot_id':uniprot_ids, 'O_patogenicity':O_pathogenicities, 'M_patogenicity':M_pathogenicities, 'I_patogenicity':I_pathogenicities}
avg_patho_regions_df = pd.DataFrame(data=avgs)
avg_patho_regions_df.to_csv(RES_DIR / 'pymissense/average_region_aa_pathogenicity_alphmissense.csv',index=False)
avg_patho_regions_df

,uniprot_id,O_patogenicity,M_patogenicity,I_patogenicity
0,P11166,0.498154,0.720000,0.549740
1,P11168,0.313564,0.578216,0.432468
2,P11169,0.464648,0.725393,0.516899


In [13]:
# The amino acid residues lining the pores of individual proteins were calculated using: https://mole.upol.cz/ -LINING RESIDUES
# The amino acid residues framing the protein binding site were calculated using: https://prankweb.cz/ - BINDING PLACE
# The results were transcribed into an .xlsx document. Saved here /GLUT project documentation/Binding places and lining residues/Excel files/
# After transcription, a third column was created, where only those amino acid residues that were not part of the binding site but framed the protein pore were transcribed. -BINDING PLACE-LINING RESIDUES